In [ ]:
import sys
from pathlib import Path

ruta_raiz = Path.cwd().parent
ruta_de_src = Path(ruta_raiz / "src")
sys.path.append(str(ruta_de_src))

from logger import obtener_fecha_hora, registrar_operacion
from config import CONFIGURACION_DATASETS

ruta_iadiza = Path(ruta_raiz / "raw_datasets" / "Colección Ornitológica del IADIZA-CCT CONICET Mendoza IADIZA-COI" / "occurrence.txt")
ruta_xeno = Path(ruta_raiz / "raw_datasets" / "xeno-canto-bird-sounds-filtered" / "Occurrence.txt")
ruta_inat = Path(ruta_raiz / "raw_datasets" / "inaturalist-filtered" / "observations.csv")

ruta_iadiza_procesada = Path(ruta_raiz / "processed_datasets" / "occurrence_actualizado.txt")
ruta_procesada_xeno = Path(ruta_raiz / "processed_datasets" / "xeno_canto" / "Occurrence.txt")
ruta_procesada_inat = Path(ruta_raiz / "processed_datasets" / "inaturalist" / "observations.csv")

ruta_log = ruta_raiz / "logs" / "operations.log"

Ejercicio 7.A - Obtener fecha y hora actual.

In [ ]:
fecha_hora = obtener_fecha_hora()
print(f"Fecha y hora actual: {fecha_hora}")

Ejercicio 7.B - Escribir en el log.

In [ ]:
#Registramos operación de prueba
nombre_dataset = "IADIZA"

registrar_operacion(nombre_dataset, "INSERT", 3)
registrar_operacion(nombre_dataset, "UPDATE", 1)
registrar_operacion(nombre_dataset, "DELETE", 0, error=True)

# Verificamos que se escribió correctamente
ruta_log = ruta_raiz / "logs" / "operations.log"
with open(ruta_log, "r", encoding="utf-8") as log:
    print(log.read())

Ejercicio 7.C - Logging de inserción

In [ ]:
from insercion_registros import insertar_registro, crear_estructura_registro_iadiza, crear_estructura_registro_xeno_canto, crear_estructura_registro_inaturalist, preparar_estructura_iadiza, preparar_estructura_xeno_canto, preparar_estructura_inaturalist, validar_registro_iadiza, validar_registro_xeno_canto, validar_registro_inaturalist

# IADIZA
registro_iadiza = crear_estructura_registro_iadiza(scientificName="Milvago chimango", countryCode="AR", license="CC_BY_NC_4_0")
insertar_registro(ruta_iadiza, ruta_iadiza_procesada, registro_iadiza, "IADIZA",
    preparar_fn=lambda r, ruta, cols: preparar_estructura_iadiza(r, ruta, cols, "IADIZA"),
    validar_fn=validar_registro_iadiza)

# Xeno-canto
registro_xeno = crear_estructura_registro_xeno_canto(scientificName="Elaenia chilensis", country="Argentina", license="CC BY-NC")
insertar_registro(ruta_xeno, ruta_procesada_xeno, registro_xeno, "xeno_canto",
    preparar_fn=preparar_estructura_xeno_canto, separador=",", validar_fn=validar_registro_xeno_canto)

# iNaturalist
registro_inat = crear_estructura_registro_inaturalist(scientificName="Thamnophilus doliatus", countryCode="AR", license="CC BY 4.0")
insertar_registro(ruta_inat, ruta_procesada_inat, registro_inat, "inaturalist",
    preparar_fn=preparar_estructura_inaturalist, separador=",", validar_fn=validar_registro_inaturalist)

print("--- Log después de INSERT ---")
with open(ruta_log, "r", encoding="utf-8") as log:
    print(log.read())

Ejercicio 7.D - Logging de actualización

In [ ]:
from actualizacion_registros import actualizar_un_campo_iadiza, actualizar_un_campo_xeno_canto, actualizar_un_campo_inaturalist

# IADIZA
actualizar_un_campo_iadiza(ruta_iadiza, ruta_iadiza_procesada, "IADIZA:COI:006869", "stateProvince", "Mendoza")

# Xeno-canto
actualizar_un_campo_xeno_canto(ruta_xeno, ruta_procesada_xeno, "https://data.biodiversitydata.nl/xeno-canto/observation/XC999540", "country", "Argentina")

# iNaturalist
actualizar_un_campo_inaturalist(ruta_inat, ruta_procesada_inat, "https://www.inaturalist.org/observations/42369866", "stateProvince", "Buenos Aires")

print("--- Log después de UPDATE ---")
with open(ruta_log, "r", encoding="utf-8") as log:
    print(log.read())

Ejercicio 7.E - Logging de eliminación

In [ ]:
from eliminacion_registros import eliminar_por_id_iadiza, eliminar_por_id_xeno_canto, eliminar_por_id_inaturalist

# IADIZA
eliminar_por_id_iadiza(ruta_iadiza, ruta_iadiza_procesada, "occurrenceID", "IADIZA:COI:006861")

# Xeno-canto
eliminar_por_id_xeno_canto(ruta_xeno, ruta_procesada_xeno, "occurrenceID", "https://data.biodiversitydata.nl/xeno-canto/observation/XC999540")

# iNaturalist
eliminar_por_id_inaturalist(ruta_inat, ruta_procesada_inat, "occurrenceID", "https://www.inaturalist.org/observations/42369866")

print("--- Log después de DELETE ---")
with open(ruta_log, "r", encoding="utf-8") as log:
    print(log.read())

Ejercicio 7.F - Logging de error

In [ ]:
# IADIZA - registro inválido
registro_invalido_iadiza = crear_estructura_registro_iadiza(decimalLatitude="999", scientificName="Milvago chimango")
insertar_registro(ruta_iadiza, ruta_iadiza_procesada, registro_invalido_iadiza, "IADIZA",
    preparar_fn=lambda r, ruta, cols: preparar_estructura_iadiza(r, ruta, cols, "IADIZA"),
    validar_fn=validar_registro_iadiza)

# Xeno-canto - registro inválido
registro_invalido_xeno = crear_estructura_registro_xeno_canto(latitudeDecimal="999", scientificName="Elaenia chilensis")
insertar_registro(ruta_xeno, ruta_procesada_xeno, registro_invalido_xeno, "xeno_canto",
    preparar_fn=preparar_estructura_xeno_canto, separador=",", validar_fn=validar_registro_xeno_canto)

# iNaturalist - registro inválido
registro_invalido_inat = crear_estructura_registro_inaturalist(decimalLatitude="999", scientificName="Thamnophilus doliatus")
insertar_registro(ruta_inat, ruta_procesada_inat, registro_invalido_inat, "inaturalist",
    preparar_fn=preparar_estructura_inaturalist, separador=",", validar_fn=validar_registro_inaturalist)

print("--- Log después de ERROR ---")
with open(ruta_log, "r", encoding="utf-8") as log:
    print(log.read())